# PRP Adjudication Error-Bar Analysis

This notebook analyzes `prp_adjudication_df` from `adjudicated_prp_affinities.csv`.

It provides:
- Error bars for each (Abstract, PRP) based on `Affinity_Std`
- Affinity-level plots with error bars per PRP
- Counts by affinity level per PRP
- Propagated count uncertainty using Monte Carlo sampling from row-wise affinity uncertainties

In [23]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

In [24]:
def affinity_level_fixed(score):
    if pd.isna(score):
        return pd.NA
    try:
        x = float(score)
    except Exception:
        return pd.NA
    if 0 <= x <= 40:
        return 'Low'
    if 40 < x <= 70:
        return 'Moderate'
    if 70 < x <= 100:
        return 'High'
    return pd.NA

In [25]:

# Locate latest benchmark run
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'
run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
run_dir = run_dirs[-1]
# Load RA-to-PRP mapping from processed CSV (generated by scripts/prepare_data.py)
mapping_path = repo_root / 'data' / 'processed' / 'ra_grouping_ra2025.csv'
ra_grouping_df = pd.read_csv(mapping_path)

required_cols = {'RA2025', 'Grouping'}
missing_cols = required_cols - set(ra_grouping_df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns in mapping file: {missing_cols}')

ra_to_prp_map = (
    ra_grouping_df[['RA2025', 'Grouping']]
    .dropna(subset=['RA2025', 'Grouping'])
    .drop_duplicates()
    .rename(columns={'RA2025': 'RA2025_ID', 'Grouping': 'PRP_Name'})
)
ra_to_prp_map['RA2025_ID'] = pd.to_numeric(ra_to_prp_map['RA2025_ID'], errors='coerce').astype('Int64')

print('Mapping file:', mapping_path)
print('Mapping rows:', len(ra_to_prp_map))
display(ra_to_prp_map.head(10))

Mapping file: /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/processed/ra_grouping_ra2025.csv
Mapping rows: 48


,RA2025_ID,PRP_Name
0,1,IBR
1,2,IBR
2,3,IBR
3,4,IBR
4,5,IBR
5,6,IBR
6,7,Stability
7,8,Stability
8,9,Stability
9,10,Stability


## RA to PRP Aggregation and Affinity Counts

This section maps `RA2025_ID` to `PRP_Name`, collapses repeated `(Abstract_Index, PRP_Name)` rows by taking the maximum `LLM_Affinity`, and then plots deterministic affinity-level counts by PRP.


In [26]:
# Locate latest benchmark run
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'
run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
run_dir = run_dirs[-1]

ra_path_adjudication = run_dir / 'adjudicated_ra_affinities.csv'
ra_adjudication_df = pd.read_csv(ra_path_adjudication)

print('Run directory:', run_dir)
print('Rows:', len(ra_adjudication_df))
display(ra_adjudication_df.head(10))

Run directory: /mnt/c/Users/pgallego/OneDrive - Imperial College London/PhD/Repositories/article-classifier/data/results/affinity_benchmark/20260413_151017
Rows: 4791


,Abstract_Index,Document Title,RA2025_ID,RA_Question,N_Models,Affinity_Mean,Affinity_Std,Affinity_Min,Affinity_Max,Affinity_Range,agreement_mean,agreement_std,agreement_min,agreement_max,Final_Affinity,Final_Reason,Final_Method,Model_Slug,Cosine_x100_Mean,Run_ID
0,1,Scenario Reduction Network Based on Wasserstei...,1,what is the future of active power balance as ...,4,6.25,12.500000,0.0,25.0,25.0,1.0,0.000000,1.00,1.00,6.25,NaN,ensemble_mean,NaN,NaN,20260413_151017
1,1,Scenario Reduction Network Based on Wasserstei...,2,how will protection systems need to change to ...,4,5.00,10.000000,0.0,20.0,20.0,1.0,0.000000,1.00,1.00,5.00,NaN,ensemble_mean,NaN,NaN,20260413_151017
2,1,Scenario Reduction Network Based on Wasserstei...,3,how could the ibr control paradigm change from...,4,7.50,15.000000,0.0,30.0,30.0,1.0,0.000000,1.00,1.00,7.50,NaN,ensemble_mean,NaN,NaN,20260413_151017
3,1,Scenario Reduction Network Based on Wasserstei...,4,what are the appropriate inverter capabilities...,4,8.75,17.500000,0.0,35.0,35.0,1.0,0.000000,1.00,1.00,8.75,NaN,ensemble_mean,NaN,NaN,20260413_151017
4,1,Scenario Reduction Network Based on Wasserstei...,5,what recommendations for standard ibr frequenc...,4,5.00,10.000000,0.0,20.0,20.0,1.0,0.000000,1.00,1.00,5.00,NaN,ensemble_mean,NaN,NaN,20260413_151017
5,1,Scenario Reduction Network Based on Wasserstei...,6,what requirements (e.g. impedance) should be p...,4,3.75,7.500000,0.0,15.0,15.0,1.0,0.000000,1.00,1.00,3.75,NaN,ensemble_mean,NaN,NaN,20260413_151017
6,1,Scenario Reduction Network Based on Wasserstei...,7,what approaches can be taken to track the dyna...,4,5.00,10.000000,0.0,20.0,20.0,1.0,0.000000,1.00,1.00,5.00,NaN,ensemble_mean,NaN,NaN,20260413_151017
7,1,Scenario Reduction Network Based on Wasserstei...,8,"what methods, including data types, and analys...",4,6.25,12.500000,0.0,25.0,25.0,1.0,0.000000,1.00,1.00,6.25,NaN,ensemble_mean,NaN,NaN,20260413_151017
8,1,Scenario Reduction Network Based on Wasserstei...,9,what analytical tools and models should be pro...,4,25.00,33.166248,0.0,70.0,70.0,0.5,0.353553,0.25,0.75,28.00,There is mixed evidence: one model sees the sc...,adjudicated_llm,gpt-5.4,NaN,20260413_151017
9,1,Scenario Reduction Network Based on Wasserstei...,10,what methods and tools should be used in plann...,4,8.75,17.500000,0.0,35.0,35.0,1.0,0.000000,1.00,1.00,8.75,NaN,ensemble_mean,NaN,NaN,20260413_151017


In [27]:
# Map RA questions to PRP names using data/processed/ra_grouping_ra2025.csv, then collapse abstract/PRP pairs
ra_analysis_df = ra_adjudication_df.copy()
ra_analysis_df['RA2025_ID'] = pd.to_numeric(ra_analysis_df['RA2025_ID'], errors='coerce').astype('Int64')
ra_analysis_df['Final_Affinity'] = pd.to_numeric(ra_analysis_df['Final_Affinity'], errors='coerce')

ra_analysis_df = ra_analysis_df.merge(
    ra_to_prp_map,
    on='RA2025_ID',
    how='left'
 )

ra_analysis_df = ra_analysis_df.dropna(subset=['Abstract_Index', 'PRP_Name', 'Final_Affinity']).copy()

ra_analysis_df = (
    ra_analysis_df.groupby(['Abstract_Index', 'PRP_Name'], as_index=False)
    .agg(Final_Affinity=('Final_Affinity', 'max'))
)
ra_analysis_df['Affinity_Level'] = ra_analysis_df['Final_Affinity'].apply(affinity_level_fixed)

print('Prepared rows:', len(ra_analysis_df))
display(ra_analysis_df.head(10))

Prepared rows: 599


,Abstract_Index,PRP_Name,Final_Affinity,Affinity_Level
0,1,CROF,22.00,Low
1,1,DER,12.50,Low
2,1,IBR,8.75,Low
3,1,Planning,76.00,High
4,1,Stability,28.00,Low
5,1,System Services,20.00,Low
6,2,CROF,31.00,Low
7,2,DER,2.50,Low
8,2,IBR,24.00,Low
9,2,Planning,8.25,Low


In [28]:
# Deterministic counts by mapped PRP name and affinity level
ra_count_df = (
    ra_analysis_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['Abstract_Index']
    .nunique()
    .rename(columns={'Abstract_Index': 'n_abstracts'})
)

fig = px.bar(
    ra_count_df,
    x='PRP_Name',
    y='n_abstracts',
    color='Affinity_Level',
    barmode='group',
    category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
    title='RA-to-PRP Abstract Counts by Affinity Level and PRP'
)
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Number of Abstracts', legend_title='Affinity Level')
fig.update_xaxes(tickangle=45)
fig.show()

display(ra_count_df.sort_values(['PRP_Name', 'Affinity_Level']))

,PRP_Name,Affinity_Level,n_abstracts
0,CROF,High,15
1,CROF,Low,39
2,CROF,Moderate,45
3,DER,High,9
4,DER,Low,80
5,DER,Moderate,11
6,IBR,High,10
7,IBR,Low,75
8,IBR,Moderate,15
9,Planning,High,21


In [29]:
# Overlap analysis across PRP areas
## 1) Pairwise overlap heatmap (Jaccard) using a configurable high-affinity threshold
## 2) Distribution plot (violin) of Final_Affinity by PRP
## 3) Scatter matrix on per-abstract PRP affinities (works best for a small number of PRPs)

high_affinity_threshold = 40.0

# Build abstract x PRP matrix from the already aggregated ra_analysis_df
wide_affinity = (
    ra_analysis_df
    .pivot_table(index='Abstract_Index', columns='PRP_Name', values='Final_Affinity', aggfunc='max')
    .sort_index(axis=1)
 )

# Binary membership: whether each abstract is strongly associated with each PRP
membership = (wide_affinity >= high_affinity_threshold).fillna(False).astype(int)

# Pairwise Jaccard overlap = |A∩B| / |A∪B|
prp_cols = membership.columns.tolist()
jaccard = pd.DataFrame(index=prp_cols, columns=prp_cols, dtype=float)

for a in prp_cols:
    A = membership[a] == 1
    for b in prp_cols:
        B = membership[b] == 1
        union = (A | B).sum()
        inter = (A & B).sum()
        jaccard.loc[a, b] = (inter / union) if union > 0 else np.nan

fig = px.imshow(
    jaccard,
    text_auto='.2f',
    color_continuous_scale='Blues',
    zmin=0,
    zmax=1,
    title=f'Pairwise PRP Overlap (Jaccard) at Final_Affinity >= {high_affinity_threshold}'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='PRP Name')
fig.show()

# Optional: raw intersection counts (how many abstracts overlap between PRP pairs)
intersections = pd.DataFrame(index=prp_cols, columns=prp_cols, dtype=float)
for a in prp_cols:
    A = membership[a] == 1
    for b in prp_cols:
        B = membership[b] == 1
        intersections.loc[a, b] = float((A & B).sum())

fig = px.imshow(
    intersections,
    text_auto='.0f',
    color_continuous_scale='Viridis',
    title=f'Pairwise PRP Intersection Counts at Final_Affinity >= {high_affinity_threshold}'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='PRP Name')
fig.show()

# Distribution overlap of affinities per PRP
fig = px.violin(
    ra_analysis_df,
    x='PRP_Name',
    y='Final_Affinity',
    color='PRP_Name',
    box=True,
    points='all',
    title='Distribution Overlap of Final_Affinity by PRP'
 )
fig.update_layout(xaxis_title='PRP Name', yaxis_title='Final Affinity', showlegend=False)
fig.update_xaxes(tickangle=45)
fig.show()

# Scatter matrix (quick exploratory view of pairwise relationships between PRPs)
# Note: can get cluttered if the number of PRPs grows.
scatter_df = wide_affinity.reset_index(drop=True).copy()

fig = px.scatter_matrix(
    scatter_df,
    dimensions=prp_cols,
    title='Scatter Matrix of PRP Affinities (per abstract)'
 )
fig.update_traces(diagonal_visible=False, marker=dict(size=5, opacity=0.45))
fig.show()

print('Abstract x PRP matrix shape:', wide_affinity.shape)
display(wide_affinity.head(10))
display(jaccard)

Abstract x PRP matrix shape: (100, 6)


PRP_Name,CROF,DER,IBR,Planning,Stability,System Services
Abstract_Index,,,,,,
1,22.0,12.5,8.75,76.00,28.000000,20.0
2,31.0,2.5,24.00,8.25,78.000000,36.0
3,20.0,30.0,12.00,72.00,13.333333,60.0
4,74.0,46.0,26.00,38.00,20.000000,77.0
5,38.0,8.0,10.00,20.00,28.000000,22.0
6,52.0,0.0,0.00,58.00,6.000000,44.0
7,76.0,12.5,3.75,68.00,11.250000,24.0
8,60.0,10.0,21.25,73.00,84.000000,26.0
9,58.0,7.5,60.00,56.00,62.000000,62.0


,CROF,DER,IBR,Planning,Stability,System Services
CROF,1.000000,0.169014,0.225352,0.405797,0.368421,0.416667
DER,0.169014,1.000000,0.210526,0.098039,0.105263,0.220000
IBR,0.225352,0.210526,1.000000,0.034483,0.367347,0.477273
Planning,0.405797,0.098039,0.034483,1.000000,0.084507,0.190476
Stability,0.368421,0.105263,0.367347,0.084507,1.000000,0.413793
System Services,0.416667,0.220000,0.477273,0.190476,0.413793,1.000000
